In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import sklearn as skl;

In [2]:
train_data = pd.read_csv('../mitsui-commodity-prediction-challenge/train.csv')
test_data = pd.read_csv('../mitsui-commodity-prediction-challenge/test.csv')
pairs = pd.read_csv('../mitsui-commodity-prediction-challenge/target_pairs.csv')
labels = pd.read_csv('../mitsui-commodity-prediction-challenge/train_labels.csv')

In [4]:
train_data.describe()

,date_id,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,...,FX_GBPCAD,FX_CADCHF,FX_NZDCAD,FX_NZDCHF,FX_ZAREUR,FX_NOKGBP,FX_NOKCHF,FX_ZARCHF,FX_NOKJPY,FX_ZARGBP
count,1917.000000,1867.000000,1867.000000,1867.000000,1867.000000,1802.000000,1802.000000,1802.000000,1802.000000,1802.000000,...,1917.000000,1917.000000,1917.000000,1917.000000,1917.000000,1917.000000,1917.000000,1917.000000,1917.000000,1917.000000
mean,958.000000,2245.249839,7886.749031,2087.633251,2796.914419,7502.445061,7561.660932,7502.554384,3749.656770,3750.703108,...,1.707502,0.705695,0.856849,0.605530,0.056180,0.082579,0.099816,0.059007,13.067047,0.048746
std,553.534552,400.328518,1515.500052,184.817440,449.827605,2833.109824,2890.201926,2833.227374,669.049121,670.412145,...,0.064488,0.046938,0.036265,0.057215,0.005703,0.006941,0.013789,0.009728,0.959581,0.005288
min,0.000000,1462.000000,4630.000000,1585.500000,1815.500000,4171.000000,4216.000000,4171.000000,2200.000000,2164.000000,...,1.472061,0.584841,0.767636,0.466372,0.045552,0.069614,0.076132,0.042087,9.618859,0.039464
25%,479.000000,1911.250000,6361.500000,1972.000000,2476.750000,5189.250000,5204.000000,5189.250000,3144.500000,3148.250000,...,1.682732,0.666179,0.827640,0.549894,0.050828,0.075427,0.084490,0.049130,12.528221,0.043534
50%,958.000000,2236.500000,8186.500000,2075.000000,2780.000000,6575.000000,6619.000000,6574.500000,3779.000000,3781.500000,...,1.712607,0.722136,0.854551,0.617905,0.056236,0.083859,0.101772,0.058538,13.231644,0.048767
75%,1437.000000,2496.000000,9251.250000,2191.750000,3040.500000,8857.875000,8883.000000,8859.750000,4311.500000,4311.750000,...,1.741451,0.745166,0.883845,0.649959,0.060712,0.087218,0.109541,0.066441,13.781044,0.052576
max,1916.000000,3849.000000,10889.000000,2681.000000,4498.500000,15720.000000,15984.000000,15715.000000,5399.500000,5400.000000,...,1.869329,0.788635,0.950091,0.710020,0.070469,0.094472,0.125403,0.081612,15.314668,0.062025


In [10]:
train_data.isnull().sum()
    

date_id          0
LME_AH_Close    50
LME_CA_Close    50
LME_PB_Close    50
LME_ZS_Close    50
                ..
FX_NOKGBP        0
FX_NOKCHF        0
FX_ZARCHF        0
FX_NOKJPY        0
FX_ZARGBP        0
Length: 558, dtype: int64

In [11]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(random_state=42)

imputed_train = imputer.fit_transform(train_data)

train_data = pd.DataFrame(imputed_train,columns=train_data.columns)

In [12]:
train_data

,date_id,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,...,FX_GBPCAD,FX_CADCHF,FX_NZDCAD,FX_NZDCHF,FX_ZAREUR,FX_NOKGBP,FX_NOKCHF,FX_ZARCHF,FX_NOKJPY,FX_ZARGBP
0,0.0,2264.5,7205.0,2570.0,3349.0,7501.530967,7555.927609,7503.862135,3745.332534,3749.429652,...,1.699987,0.776874,0.888115,0.689954,0.066653,0.090582,0.119630,0.078135,13.822740,0.059163
1,1.0,2228.0,7147.0,2579.0,3327.0,7502.968199,7556.154714,7503.465496,3745.377415,3750.254625,...,1.695279,0.778682,0.889488,0.692628,0.067354,0.091297,0.120520,0.079066,13.888146,0.059895
2,2.0,2250.0,7188.5,2587.0,3362.0,4684.000000,4691.000000,4684.000000,3363.000000,3367.000000,...,1.692724,0.780186,0.894004,0.697490,0.067394,0.091478,0.120809,0.079287,13.983675,0.060037
3,3.0,2202.5,7121.0,2540.0,3354.0,4728.000000,4737.000000,4729.000000,3430.000000,3426.000000,...,1.683111,0.785329,0.889439,0.698502,0.067639,0.091558,0.121021,0.079285,14.035571,0.059983
4,4.0,2175.0,7125.0,2604.0,3386.0,7502.808706,7554.196332,7504.085808,3747.110378,3750.349449,...,1.684816,0.787264,0.891042,0.701485,0.067443,0.091266,0.121055,0.078925,14.013760,0.059503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1912,1912.0,2450.0,9523.5,1961.5,2676.5,15086.000000,15440.000000,15085.000000,4461.500000,4467.000000,...,1.864661,0.598318,0.827529,0.495125,0.049224,0.072574,0.080968,0.046175,14.058107,0.041388
1913,1913.0,2471.5,9519.5,1980.5,2710.5,15165.000000,15509.000000,15162.000000,4495.000000,4490.000000,...,1.863539,0.594400,0.824390,0.490018,0.049409,0.072828,0.080671,0.046113,14.082236,0.041630
1914,1914.0,2471.5,9533.5,1974.0,2693.0,15040.000000,15477.000000,15044.000000,4544.500000,4555.000000,...,1.860067,0.595250,0.822392,0.489529,0.049095,0.073232,0.081083,0.045901,14.126606,0.041457
1915,1915.0,2456.0,9500.5,1970.0,2697.5,15420.000000,15752.000000,15420.000000,4670.000000,4685.000000,...,1.859624,0.597780,0.817224,0.488520,0.049205,0.073018,0.081170,0.045987,14.095322,0.041368


In [13]:
train_data.to_csv("../mitsui-commodity-prediction-challenge/imputed_data.csv", index = False)